# E021 — Audio PLP (Perceptual Linear Prediction)

Replaces MFCC with PLP (Hermansky 1990) — psychoacoustically motivated features
combining Bark-scale filterbank, equal loudness weighting, cube-root compression,
and LPC cepstrum. More robust than MFCC in channel-mismatched/noisy conditions.

**Feature dimension:** 13 static + Δ + ΔΔ = **39 features/frame** (same as E008)

Everything else identical to E008 +All:
- UBM 32, MAP r=16
- Augmentation: noise SNR=20dB + speed ±10%
- LOSO 3-fold, seed=67

Reference EERs: E008 +All (MFCC): fold0=3.47, fold1=8.33, fold2=0.83, mean=4.21±3.11%  
Reference EERs: E020 +All (LPCC): fold0=9.17, fold1=0.83, fold2=0.00, mean=3.33±4.14%

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve, auc
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
from scipy.linalg import solve_toeplitz
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
CONFIG_COLORS = {
    "E008 +All (ref)": "#95A5A6",
    "E020 LPCC (ref)": "#E67E22",
    "PLP +All":        "#E74C3C",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. PLP feature extraction

PLP (Hermansky 1990) pipeline:

1. **Bark filterbank** — triangular filters on the Bark scale (mimics cochlear frequency resolution)
2. **Equal loudness weighting** — weights filterbank outputs by the ear's sensitivity curve
3. **Cube-root compression** — intensity → loudness (less saturating than log)
4. **LPC analysis** — all-pole model fit to the compressed spectrum
5. **Cepstral recursion** — convert LPC coefficients to cepstrum

With n_barks=20, LPC order=12, and n_cep=13 we obtain 13+Δ+ΔΔ = 39d (same as E008).

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def hz_to_bark(f):
    return 6.0 * np.arcsinh(f / 600.0)


def bark_to_hz(b):
    return 600.0 * np.sinh(b / 6.0)


def equal_loudness_weight(f):
    """Simplified equal loudness curve (Hermansky 1990)."""
    f = np.asarray(f, dtype=float)
    f2 = f * f
    f4 = f2 * f2
    num = (f2 + 1.44e6) * f4
    den = ((f2 + 1.6e9) ** 2) * (f2 + 9.61e6)
    return num / (den + 1e-30)


def extract_plp(y: np.ndarray, sr: int, order: int = 12, n_cep: int = 13,
                n_barks: int = 20, hop_length: int = 160,
                win_length: int = 400) -> np.ndarray:
    """PLP features: Bark filterbank + equal loudness + cube root + LPC + cepstrum."""
    n_fft = 512
    # Power spectrogram
    D = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length,
                             win_length=win_length)) ** 2  # (n_fft//2+1, T)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)  # (n_fft//2+1,)

    # Bark filter bank: n_barks triangular filters on Bark scale
    bark_max = hz_to_bark(sr / 2)
    bark_centers = np.linspace(0, bark_max, n_barks + 2)
    hz_centers = bark_to_hz(bark_centers)

    # Build triangular filters in Hz
    filterbank = np.zeros((n_barks, len(freqs)))
    for i in range(n_barks):
        lo, center, hi = hz_centers[i], hz_centers[i+1], hz_centers[i+2]
        for j, f in enumerate(freqs):
            if lo <= f <= center:
                filterbank[i, j] = (f - lo) / (center - lo + 1e-10)
            elif center < f <= hi:
                filterbank[i, j] = (hi - f) / (hi - center + 1e-10)

    # Apply equal loudness to filterbank center frequencies
    el = equal_loudness_weight(hz_centers[1:-1])  # (n_barks,)
    filterbank = filterbank * el[:, None]

    # Filter bank output: (n_barks, T)
    bark_spec = filterbank @ D

    # Cube root compression (intensity-loudness)
    bark_spec = np.cbrt(bark_spec + 1e-30)   # (n_barks, T)

    # Per-frame LPC + cepstrum
    T = bark_spec.shape[1]
    plp_frames = []
    for t in range(T):
        spec = bark_spec[:, t]
        # Autocorrelation from spectrum (via IDFT of power spectrum)
        # Symmetric spectrum for IFFT
        sym = np.concatenate([spec, spec[-2:0:-1]])
        r = np.real(np.fft.ifft(sym))[:order+1]
        # Solve Yule-Walker for LPC
        try:
            R = r[:order]
            a_lpc = solve_toeplitz(R, -r[1:order+1])
            a = np.concatenate([[1.0], a_lpc])
        except Exception:
            a = np.zeros(order + 1); a[0] = 1.0
        # Cepstrum from LPC via frequency domain
        A_freq = np.fft.rfft(a, n=512)
        log_H = -np.log(np.abs(A_freq) + 1e-10)
        cep = np.real(np.fft.irfft(log_H))[:n_cep]
        plp_frames.append(cep)

    plp = np.array(plp_frames, dtype=np.float32)   # (T, 13)
    delta  = librosa.feature.delta(plp.T).T
    delta2 = librosa.feature.delta(plp.T, order=2).T
    feat   = np.hstack([plp, delta, delta2])         # (T, 39)
    feat  -= feat.mean(axis=0)                       # CMN
    return feat


def aug_noise(y: np.ndarray, snr_db: float = 20.0, rng: np.random.Generator = None) -> np.ndarray:
    """Add white noise at target SNR."""
    signal_power = np.mean(y ** 2) + 1e-10
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(y)).astype(y.dtype)
    return y + noise


def aug_speed(y: np.ndarray, rate_range=(0.9, 1.1), rng: np.random.Generator = None) -> np.ndarray:
    """Random time stretch without changing pitch."""
    rate = rng.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)


def load_and_augment(wav_path: Path, rng: np.random.Generator):
    """
    Load WAV and return list of (y, sr) tuples: [original, noisy, speed-perturbed].
    Matches E008 +All augmentation.
    """
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [
        (y, sr),
        (aug_noise(y, snr_db=20.0, rng=rng), sr),
        (aug_speed(y, rng=rng), sr),
    ]


def extract_batch(df: pd.DataFrame, data_dir: Path, seed: int):
    """Extract PLP features for all samples with +All augmentation."""
    rng = np.random.default_rng(seed)
    all_feat, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in load_and_augment(find_wav(row["stem"], data_dir), rng):
            feat = extract_plp(y_aug, sr)
            all_feat.append(feat)
            all_labels.extend([row["label"]] * len(feat))
    return np.vstack(all_feat), np.array(all_labels)


def score_utterance_plp(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat  = extract_plp(y, sr)
    return float((adapted.score_samples(feat) - ubm.score_samples(feat)).mean())


# Sanity check dimensions
sample_row = manifest[manifest.label == 1].iloc[0]
y_tmp, sr_tmp = librosa.load(find_wav(sample_row["stem"], DATA), sr=None, mono=True)
feat_tmp = extract_plp(y_tmp, sr_tmp)
print(f"PLP feature shape: {feat_tmp.shape}  (T frames × 39 dims)")
print(f"  static PLP: 13 dims, delta: 13, delta2: 13, total: 39")
print(f"  Same dimensionality as E008 MFCC+Δ+ΔΔ and E020 LPCC+Δ+ΔΔ")
print("Feature extraction functions defined.")

## 2. PLP filterbank visualization

Visualise the Bark-scale triangular filters and equal loudness weights,
then compare PLP vs MFCC feature matrices for a target sample.

In [ ]:
# --- Visualise Bark filterbank and equal loudness curve ---
n_fft = 512
freqs = librosa.fft_frequencies(sr=sr_tmp, n_fft=n_fft)
bark_max = hz_to_bark(sr_tmp / 2)
bark_centers = np.linspace(0, bark_max, 22)
hz_centers = bark_to_hz(bark_centers)

filterbank_viz = np.zeros((20, len(freqs)))
for i in range(20):
    lo, center, hi = hz_centers[i], hz_centers[i+1], hz_centers[i+2]
    for j, f in enumerate(freqs):
        if lo <= f <= center:
            filterbank_viz[i, j] = (f - lo) / (center - lo + 1e-10)
        elif center < f <= hi:
            filterbank_viz[i, j] = (hi - f) / (hi - center + 1e-10)

el_curve = equal_loudness_weight(freqs + 1e-3)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for i in range(20):
    ax.plot(freqs, filterbank_viz[i], alpha=0.7)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Filter weight")
ax.set_title("PLP Bark-scale triangular filterbank (20 filters)")

ax = axes[1]
ax.plot(freqs, el_curve / (el_curve.max() + 1e-30), color=COLORS["target"])
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Equal loudness weight (normalised)")
ax.set_title("Equal loudness curve (Hermansky 1990)")

plt.suptitle("PLP psychoacoustic front-end", fontsize=12)
plt.tight_layout()
plt.show()

# --- Feature matrix comparison: MFCC vs PLP ---
y_viz, sr_viz = librosa.load(find_wav(sample_row["stem"], DATA), sr=None, mono=True)

# Standard MFCC+Δ+ΔΔ (39d)
mfcc_raw = librosa.feature.mfcc(y=y_viz, sr=sr_viz, n_mfcc=13)
delta  = librosa.feature.delta(mfcc_raw)
delta2 = librosa.feature.delta(mfcc_raw, order=2)
mfcc_39 = np.vstack([mfcc_raw, delta, delta2]).T
mfcc_39 -= mfcc_39.mean(axis=0)

# PLP+Δ+ΔΔ (39d)
plp_39 = extract_plp(y_viz, sr_viz)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(mfcc_39.T, aspect="auto", origin="lower", cmap="magma")
ax.axhline(12.5, color="white", lw=1, ls="--", alpha=0.6)
ax.axhline(25.5, color="white", lw=1, ls="--", alpha=0.6)
ax.set_title(f"MFCC + Δ + ΔΔ (39d)\nstatic | delta | delta2", fontsize=10)
ax.set_xlabel("Frame")
ax.set_ylabel("Feature dim")
plt.colorbar(im, ax=ax, shrink=0.8)

ax = axes[1]
im = ax.imshow(plp_39.T, aspect="auto", origin="lower", cmap="magma")
ax.axhline(12.5, color="white", lw=1, ls="--", alpha=0.6)
ax.axhline(25.5, color="white", lw=1, ls="--", alpha=0.6)
ax.set_title(f"PLP + Δ + ΔΔ (39d)\nstatic (Bark+EL+cbrt+LPC, order=12) | delta | delta2", fontsize=10)
ax.set_xlabel("Frame")
ax.set_ylabel("Feature dim")
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f"Feature comparison — {sample_row['stem']} (target)",
             color=COLORS["target"], fontsize=12)
plt.tight_layout()
plt.show()
print(f"Both: 39 dims/frame — same shape, different spectral representation")

## 3. UBM + MAP functions

In [ ]:
def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    log_prob   = ubm._estimate_log_prob(X_target)
    log_resp   = log_prob + np.log(ubm.weights_)
    log_resp  -= logsumexp(log_resp, axis=1, keepdims=True)
    resp       = np.exp(log_resp)
    n_k        = resp.sum(axis=0)
    mu_hat     = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha      = n_k / (n_k + r)
    adapted    = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


print("UBM+MAP functions defined.")

## 4. LOSO cross-validation — PLP +All

Train UBM and MAP on augmented (+All) PLP features from the train fold.
Val utterances always scored from original WAVs (no aug leakage).

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0

# Reference results
E008_REF  = {"fold_eers": [3.47, 8.33, 0.83], "mean": 4.21, "std": 3.11, "min_dcf_mean": 0.0509}
E020_REF  = {"fold_eers": [9.17, 0.83, 0.00], "mean": 3.33, "std": 4.14, "min_dcf_mean": 0.0333}

print("Running LOSO CV — PLP +All (39d features, UBM32, MAP r=16)")
print("=" * 60)

oof_scores   = np.full(len(manifest), np.nan)
fold_results = []

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]

    print(f"\nFold {fold_id}: {len(train_df)} train, {len(val_df)} val")

    # Augmented PLP features — train fold only
    X_train, y_train = extract_batch(train_df, DATA, seed=SEED + fold_id)
    X_nt = X_train[y_train == 0]
    X_t  = X_train[y_train == 1]

    print(f"  {len(X_t)} target frames, {len(X_nt)} non-target frames (3× aug)")
    print(f"  Feature dim: {X_train.shape[1]}")

    ubm     = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
    adapted = map_adapt(ubm, X_t, r=MAP_R)

    # Score val on ORIGINAL WAVs only
    for idx, row in val_df.iterrows():
        oof_scores[idx] = score_utterance_plp(find_wav(row["stem"], DATA), adapted, ubm)

    val_scores = oof_scores[val_idx]
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    eer, _     = compute_eer(val_scores[val_labels==1], val_scores[val_labels==0])
    min_dcf, _ = compute_min_dcf(val_scores[val_labels==1], val_scores[val_labels==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})

    ref_e008 = E008_REF["fold_eers"][fold_id]
    ref_e020 = E020_REF["fold_eers"][fold_id]
    print(f"  → EER={eer*100:.2f}%  min-DCF={min_dcf:.4f}")
    print(f"     vs E008 MFCC: {(eer*100 - ref_e008):+.2f}pp | vs E020 LPCC: {(eer*100 - ref_e020):+.2f}pp")

print("\n" + "=" * 60)
print("CV complete.")

## 5. Results table

In [ ]:
eers_plp  = [r["eer"]*100  for r in fold_results]
dcfs_plp  = [r["min_dcf"] for r in fold_results]
mean_plp  = np.mean(eers_plp)
std_plp   = np.std(eers_plp)
mean_dcf  = np.mean(dcfs_plp)

print(f"{'Config':<22} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 76)
ref = E008_REF
print(f"{'E008 +All (ref MFCC)':<22} {ref['fold_eers'][0]:>8.2f} {ref['fold_eers'][1]:>8.2f} "
      f"{ref['fold_eers'][2]:>8.2f} {ref['mean']:>8.2f} {ref['std']:>8.2f} {ref['min_dcf_mean']:>9.4f}")
ref2 = E020_REF
print(f"{'E020 LPCC (ref)':<22} {ref2['fold_eers'][0]:>8.2f} {ref2['fold_eers'][1]:>8.2f} "
      f"{ref2['fold_eers'][2]:>8.2f} {ref2['mean']:>8.2f} {ref2['std']:>8.2f} {ref2['min_dcf_mean']:>9.4f}")
print(f"{'PLP +All':<22} {eers_plp[0]:>8.2f} {eers_plp[1]:>8.2f} {eers_plp[2]:>8.2f} "
      f"{mean_plp:>8.2f} {std_plp:>8.2f} {mean_dcf:>9.4f}")
print("-" * 76)

delta_e008 = mean_plp - E008_REF["mean"]
delta_e020 = mean_plp - E020_REF["mean"]
dir_e008 = "improvement" if delta_e008 < 0 else "regression"
dir_e020 = "improvement" if delta_e020 < 0 else "regression"
print(f"\nPLP +All vs E008 (MFCC) +All: {delta_e008:+.2f}pp ({dir_e008})")
print(f"PLP +All vs E020 (LPCC) +All: {delta_e020:+.2f}pp ({dir_e020})")

eer_oof, _   = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
print(f"OOF overall: EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

## 6. Per-fold bar chart + mean ± std

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs_bar = {
    "E008 +All (ref)": E008_REF["fold_eers"],
    "E020 LPCC (ref)": E020_REF["fold_eers"],
    "PLP +All":        eers_plp,
}

# Per-fold grouped bars
ax = axes[0]
x = np.arange(3)
width = 0.25
for i, (cname, fold_eers) in enumerate(configs_bar.items()):
    offset = (i - 1) * width
    ax.bar(x + offset, fold_eers, width, label=cname,
           color=CONFIG_COLORS[cname], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("Per-fold EER — PLP vs MFCC vs LPCC")
ax.legend(fontsize=9)

# Mean ± std
ax = axes[1]
means_bar = [E008_REF["mean"], E020_REF["mean"], mean_plp]
stds_bar  = [E008_REF["std"],  E020_REF["std"],  std_plp]
cnames    = list(configs_bar.keys())
colors_b  = [CONFIG_COLORS[c] for c in cnames]
bars = ax.bar(range(3), means_bar, color=colors_b, alpha=0.85,
              yerr=stds_bar, capsize=6, error_kw=dict(elinewidth=2))
for bar, m, s in zip(bars, means_bar, stds_bar):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.15,
            f"{m:.1f}\n±{s:.1f}", ha="center", fontsize=9)
best_idx = int(np.argmin(means_bar))
bars[best_idx].set_edgecolor("gold")
bars[best_idx].set_linewidth(3)
ax.annotate("★ best", xy=(best_idx, means_bar[best_idx] - stds_bar[best_idx] - 0.3),
            ha="center", fontsize=9, color="goldenrod", fontweight="bold")
ax.set_xticks(range(3))
ax.set_xticklabels(cnames, fontsize=8)
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("Mean ± std — PLP vs references")

plt.suptitle("E021 — PLP+Δ+ΔΔ vs E008 MFCC vs E020 LPCC", fontsize=13)
plt.tight_layout()
plt.show()

## 7. DET curve

In [ ]:
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos    = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

fig, ax = plt.subplots(figsize=(7, 6))

# PLP OOF DET
valid = ~np.isnan(oof_scores)
fpr, tpr, _ = roc_curve(y_all[valid], oof_scores[valid])
far_c = np.clip(fpr, 1e-4, 1-1e-4)
frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
        color=CONFIG_COLORS["PLP +All"], lw=2.5,
        label=f"PLP +All  EER={eer_oof*100:.1f}%", zorder=5)

# Reference E008 EER point
e008_eer_norm = scipy_norm.ppf(E008_REF["mean"] / 100)
ax.scatter([e008_eer_norm], [e008_eer_norm], marker="x", s=120,
           color=CONFIG_COLORS["E008 +All (ref)"], zorder=6, lw=2,
           label=f"E008 MFCC EER={E008_REF['mean']:.2f}% (ref, point only)")

# Reference E020 EER point
e020_eer_norm = scipy_norm.ppf(E020_REF["mean"] / 100)
ax.scatter([e020_eer_norm], [e020_eer_norm], marker="^", s=120,
           color=CONFIG_COLORS["E020 LPCC (ref)"], zorder=6, lw=2,
           label=f"E020 LPCC EER={E020_REF['mean']:.2f}% (ref, point only)")

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curve — PLP +All (OOF)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Score distributions (2×2)

In [ ]:
fold_scores_list = []
for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    fold_scores_list.append({
        "scores": oof_scores[val_idx],
        "labels": manifest.loc[val_idx, "label"].to_numpy()
    })

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
finite_scores = oof_scores[~np.isnan(oof_scores)]
bin_edges = np.linspace(np.nanmin(finite_scores), np.nanmax(finite_scores), 35)

for i, (ax, fdata) in enumerate(zip(axes[:3], fold_scores_list)):
    s, l = fdata["scores"], fdata["labels"]
    eer_f, thr_f = compute_eer(s[l==1], s[l==0])
    ax.hist(s[l==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
    ax.hist(s[l==1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
    ax.axvline(thr_f, color=COLORS["green"], ls="--", lw=2, label=f"thresh={thr_f:.2f}")
    ax.set_title(f"Fold {i}  —  EER={eer_f*100:.1f}%")
    ax.set_xlabel("Score (LLR)")
    ax.legend(fontsize=8)

ax = axes[3]
ax.hist(oof_scores[y_all==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
ax.hist(oof_scores[y_all==1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
ax.axvline(thr, color=COLORS["green"], ls="--", lw=2, label=f"OOF thresh={thr:.2f}")
ax.set_title(f"OOF overall  —  EER={eer_oof*100:.1f}%")
ax.set_xlabel("Score (LLR)")
ax.legend(fontsize=8)

plt.suptitle("E021 — PLP +All score distributions", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("E021 PLP — final numbers")
print(f"{'='*60}")
print(f"PLP +All:      EER={mean_plp:.2f}±{std_plp:.2f}%  min-DCF={mean_dcf:.4f}")
print(f"E008 MFCC ref: EER=4.21±3.11%  min-DCF=0.0509")
print(f"E020 LPCC ref: EER=3.33±4.14%  min-DCF=0.0333")
print(f"Delta vs E008: {delta_e008:+.2f}pp ({dir_e008})")
print(f"Delta vs E020: {delta_e020:+.2f}pp ({dir_e020})")
print(f"OOF overall:   EER={eer_oof*100:.2f}%  min-DCF={dcf_oof:.4f}  thr={thr:.3f}")